### Capstone 2 Airbnb Data Wrangling
* each city has a separate file for 'detailed_listings'
* merge all cities into one file
* explore data
* clean data and address null values

In [1]:
import pandas as pd
import glob
import os

# folder paths
raw_data_path  = os.path.join('..', 'data', 'raw', '*.csv')
combined_data_path = os.path.join('..', 'data', 'processed', 'combined_data.csv')

# load & combine
files = glob.glob(raw_data_path)
print(f'Found {len(files)} files:')
for f in files:
    print(f'  {f}')

df_listings = pd.concat([pd.read_csv(file, low_memory=False) for file in files], ignore_index=True)
print(f'\nCombined dataframe shape: {df_listings.shape}')

# save combined file 
df_listings.to_csv(combined_data_path, index=False)
print(f'Saved → {combined_data_path}')

Found 31 files:
  ..\data\raw\asheville_listings_detailed.csv
  ..\data\raw\austin_listings_detailed.csv
  ..\data\raw\bozeman_listings_detailed.csv
  ..\data\raw\broward_listings_detailed.csv
  ..\data\raw\cambridge_listings_detailed.csv
  ..\data\raw\chicago_listings_detailed.csv
  ..\data\raw\clark_county_listings_detailed.csv
  ..\data\raw\columbus_listings_detailed.csv
  ..\data\raw\dallas_listings_detailed.csv
  ..\data\raw\dever_listings_detailed.csv
  ..\data\raw\fort_worth_listings_detailed.csv
  ..\data\raw\hawaii_listings_detailed.csv
  ..\data\raw\jersey_city_listings_detailed.csv
  ..\data\raw\los_angeles_listings_detailed.csv
  ..\data\raw\nashville_listings_detailed.csv
  ..\data\raw\newark_listings_detailed.csv
  ..\data\raw\new_orleans_listings_detailed.csv
  ..\data\raw\nyc_listings_detailed.csv
  ..\data\raw\oakland_listings_detailed.csv
  ..\data\raw\pacific_grove_listings_detailed.csv
  ..\data\raw\portland_listings_detailed.csv
  ..\data\raw\rhode_island_listings_

In [2]:
print(f'\nCombined dataframe shape: {df_listings.shape}')


Combined dataframe shape: (268387, 75)


### Explore Dataframe
* column names
* dtypes
* null values

In [3]:
df_listings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 268387 entries, 0 to 268386
Data columns (total 75 columns):
 #   Column                                        Non-Null Count   Dtype  
---  ------                                        --------------   -----  
 0   id                                            268387 non-null  int64  
 1   listing_url                                   268387 non-null  object 
 2   scrape_id                                     268387 non-null  int64  
 3   last_scraped                                  268387 non-null  object 
 4   source                                        268387 non-null  object 
 5   name                                          268370 non-null  object 
 6   description                                   265555 non-null  object 
 7   neighborhood_overview                         166746 non-null  object 
 8   picture_url                                   268386 non-null  object 
 9   host_id                                       26

## Data Type Conversions
Convert raw string columns to numeric types for modeling:
- `price`: strip `$` and `,` → float
- `host_response_rate`: strip `%` → float  
- `host_is_superhost`: map `t/f` → `1/0`
- `instant_bookable`: map `t/f` → `1/0`

In [4]:
# convert price into a float
df_listings['price'] = df_listings['price'].replace('[\$,]', '', regex=True).astype(float) 

# convert host response rate to a float
df_listings['host_response_rate'] = df_listings['host_response_rate'].replace('[\%,]', '', regex=True).astype(float)

# convert t f values into 0 or 1
df_listings['host_is_superhost'] = df_listings['host_is_superhost'].map({'t': 1, 'f': 0})

# convert t f values into 0 or 1
df_listings['instant_bookable'] = df_listings['instant_bookable'].map({'t': 1, 'f': 0})

In [5]:
#check df shape
df_listings.shape

(268387, 75)

## Update dataframe
* remove columns that don't apply to modeling
* remove columns with all null values
* bathrooms: need to check null values (may be shared/communal bathrooms, if so, will impute to 1)
* response rate: not reported for all, will remove
* listing and picture url: will remove for now, will join for later part of capstone

In [6]:
# drop these columns, 'bathrooms', 'calendar_updated', 'neighborhood_group_cleansed' all null
drop_col = ['name','listing_url','picture_url','scrape_id','last_scraped', 'source', 'description', 'neighborhood_overview', 
            'neighbourhood_cleansed','host_url', 'host_name', 'host_since', 'host_acceptance_rate', 'neighbourhood',
            'host_location', 'host_about','host_thumbnail_url', 'host_picture_url','host_neighbourhood','host_total_listings_count', 
            'host_verifications','host_has_profile_pic', 'host_identity_verified','neighbourhood_group_cleansed',
            'calendar_last_scraped','license','calculated_host_listings_count','calculated_host_listings_count_entire_homes',
            'calculated_host_listings_count_private_rooms','calculated_host_listings_count_shared_rooms','has_availability',
            'availability_30', 'availability_60', 'availability_90','maximum_nights_avg_ntm','bathrooms', 'calendar_updated',
            'minimum_nights_avg_ntm','maximum_maximum_nights','minimum_maximum_nights',
            'maximum_minimum_nights','minimum_minimum_nights','maximum_nights','minimum_nights',
            'number_of_reviews_l30d','availability_365','host_listings_count','number_of_reviews_ltm',
            'first_review','last_review','review_scores_accuracy','review_scores_cleanliness','review_scores_checkin',
            'review_scores_communication','review_scores_location','review_scores_value','host_response_rate']

In [7]:
df = df_listings.drop(drop_col, axis=1).copy()

In [8]:
# check number of columns and rows
df.shape

(268387, 18)

## Inspect Object Columns
Identify string columns that need to be converted, encoded, or dropped before modeling.

In [9]:
object_columns = df.select_dtypes(include='object')
object_columns.T

,0,1,2,3,4,5,6,7,8,9,...,268377,268378,268379,268380,268381,268382,268383,268384,268385,268386
host_response_time,within an hour,within an hour,within an hour,within a few hours,within a few hours,within a few hours,within an hour,within a day,within an hour,NaN,...,within a few hours,within a few hours,within a few hours,within a few hours,within an hour,within an hour,within a few hours,NaN,within an hour,NaN
property_type,Entire rental unit,Entire guesthouse,Private room in home,Private room in home,Entire guest suite,Private room in cabin,Entire rental unit,Entire guesthouse,Entire guest suite,Entire guest suite,...,Entire rental unit,Entire home,Private room in home,Private room in home,Entire townhouse,Entire rental unit,Entire home,Entire home,Private room in rental unit,Entire rental unit
room_type,Entire home/apt,Entire home/apt,Private room,Private room,Entire home/apt,Private room,Entire home/apt,Entire home/apt,Entire home/apt,Entire home/apt,...,Entire home/apt,Entire home/apt,Private room,Private room,Entire home/apt,Entire home/apt,Entire home/apt,Entire home/apt,Private room,Entire home/apt
bathrooms_text,1 bath,1 bath,2.5 shared baths,1 private bath,1 bath,1 shared bath,1 bath,1 bath,1 bath,1 bath,...,1 bath,3 baths,1 private bath,1 private bath,1 bath,1 bath,3.5 baths,2 baths,1 shared bath,1 bath
amenities,"[""Coffee maker"", ""Cleaning products"", ""Extra p...","[""Shampoo"", ""Hair dryer"", ""Coffee maker"", ""Pet...","[""Fire extinguisher"", ""Refrigerator"", ""Microwa...","[""Shampoo"", ""Electric stove"", ""Hair dryer"", ""C...","[""Shampoo"", ""Hair dryer"", ""Coffee maker"", ""Cle...","[""Coffee maker"", ""Extra pillows and blankets"",...","[""Coffee"", ""Hair dryer"", ""Coffee maker"", ""Clea...","[""Shampoo"", ""Hair dryer"", ""Coffee maker"", ""Ext...","[""Shampoo"", ""Hair dryer"", ""Coffee maker"", ""Ext...","[""Shampoo"", ""Hair dryer"", ""Dryer"", ""Cooking ba...",...,"[""Air conditioning"", ""TV"", ""Carbon monoxide al...","[""Fire extinguisher"", ""Dedicated workspace"", ""...","[""Fire extinguisher"", ""Dedicated workspace"", ""...","[""Fire extinguisher"", ""Dedicated workspace"", ""...","[""Air conditioning"", ""Private entrance"", ""TV"",...","[""BBQ grill"", ""Fire extinguisher"", ""Dedicated ...","[""Fire extinguisher"", ""Dedicated workspace"", ""...","[""BBQ grill"", ""Fire extinguisher"", ""Kitchen"", ...","[""Dedicated workspace"", ""Kitchen"", ""Air condit...","[""Fire extinguisher"", ""Dedicated workspace"", ""..."


## Clean Room Types ##
* check whether to use 'property_type' or 'room_type'
* determine if listing is entire home, private room, shared room, or hotel
* use 1 or 0 to 

In [10]:
#check property types
df.property_type.value_counts()

property_type
Entire rental unit             67270
Entire home                    55535
Entire condo                   35122
Private room in home           25141
Private room in rental unit    18087
                               ...  
Riad                               1
Shared room in castle              1
Shared room in nature lodge        1
Entire timeshare                   1
Private room in ice dome           1
Name: count, Length: 138, dtype: int64

In [11]:
#check room types
df["room_type"].value_counts()

room_type
Entire home/apt    201296
Private room        63622
Shared room          2450
Hotel room           1019
Name: count, dtype: int64

In [12]:
#drop property type and use room type for classification
df = df.drop('property_type', errors='ignore', axis=1)
df.head(3)

,id,host_id,host_response_time,host_is_superhost,latitude,longitude,room_type,accommodates,bathrooms_text,bedrooms,beds,amenities,price,number_of_reviews,review_scores_rating,instant_bookable,reviews_per_month
0,108061,320564,within an hour,1.0,35.60670,-82.55563,Entire home/apt,2,1 bath,1.0,1.0,"[""Coffee maker"", ""Cleaning products"", ""Extra p...",100.0,92,4.51,0,0.66
1,155305,746673,within an hour,0.0,35.57864,-82.59578,Entire home/apt,2,1 bath,1.0,1.0,"[""Shampoo"", ""Hair dryer"", ""Coffee maker"", ""Pet...",100.0,383,4.59,0,2.70
2,156805,746673,within an hour,0.0,35.57864,-82.59578,Private room,2,2.5 shared baths,1.0,1.0,"[""Fire extinguisher"", ""Refrigerator"", ""Microwa...",66.0,67,4.52,1,0.48


In [13]:
df = pd.get_dummies(df, columns=['room_type'], drop_first=True) # drops 'entire home' - if it's not a private/shared/hotel room, then assume 'entire home'

**Drop property type (too much noise), use room type for clarity**

### Clean Up Column Names 

In [14]:
# make sure all columns have conventional naming with lowercase letters and underscores instead of spaces
df.columns = df.columns.str.replace(' ', '_').str.lower()

In [15]:
df.columns.tolist()

['id',
 'host_id',
 'host_response_time',
 'host_is_superhost',
 'latitude',
 'longitude',
 'accommodates',
 'bathrooms_text',
 'bedrooms',
 'beds',
 'amenities',
 'price',
 'number_of_reviews',
 'review_scores_rating',
 'instant_bookable',
 'reviews_per_month',
 'room_type_hotel_room',
 'room_type_private_room',
 'room_type_shared_room']

### Inspect Host Response Time
* encode with 3, 2, 1, 0 with 3 being the fastest response time and 0 being the slowest.

In [16]:
#update host response time to reflect a value instead of qualitative data
mapping = {
    "within an hour": 3,
    "within a few hours": 2,
    "within a day": 1,
    "a few days or more": 0}

df["host_response_time_ord"] = df["host_response_time"].map(mapping)

df["host_response_time_ord"] = df["host_response_time_ord"].fillna(-1)

### Confirm value counts are correct and still matching

In [17]:
df["host_response_time"].value_counts()

host_response_time
within an hour        178885
within a few hours     27269
within a day           13320
a few days or more      3159
Name: count, dtype: int64

In [18]:
df["host_response_time_ord"].value_counts()

host_response_time_ord
 3.0    178885
-1.0     45754
 2.0     27269
 1.0     13320
 0.0      3159
Name: count, dtype: int64

## Inspect Bathroom Data
* remove strings, change to int
* check null values, impute as needed

In [19]:
df['bathrooms_text'].value_counts()

bathrooms_text
1 bath               121320
2 baths               48655
1 shared bath         24670
1 private bath        19786
2.5 baths             12258
                      ...  
12.5 shared baths         1
27.5 baths                1
33.5 baths                1
23 baths                  1
22 baths                  1
Name: count, Length: 70, dtype: int64

In [20]:
#convert bathroom text to a float
df["bathrooms"] = (df["bathrooms_text"]
    .str.replace("Half-bath", "0.5", regex=False)
    .str.extract(r"(\d+\.?\d*)")[0]
    .astype(float))

In [21]:
df['bathrooms'] = df['bathrooms'].fillna(1.0) # impute with 1 bathroom since listings all have access to at least 1 bathroom (shared/communal/etc)
df['bathrooms'].isnull().sum()  # check for null values

np.int64(0)

### Clean Up Dataframe

In [22]:
df.columns.tolist()

['id',
 'host_id',
 'host_response_time',
 'host_is_superhost',
 'latitude',
 'longitude',
 'accommodates',
 'bathrooms_text',
 'bedrooms',
 'beds',
 'amenities',
 'price',
 'number_of_reviews',
 'review_scores_rating',
 'instant_bookable',
 'reviews_per_month',
 'room_type_hotel_room',
 'room_type_private_room',
 'room_type_shared_room',
 'host_response_time_ord',
 'bathrooms']

In [23]:
# drop columns, don't need this info
# Option 1: Remove the assignment to df when using inplace=True
df.drop(['host_response_time', 'bathrooms_text'], axis=1, inplace=True)

In [24]:
df.columns.tolist()

['id',
 'host_id',
 'host_is_superhost',
 'latitude',
 'longitude',
 'accommodates',
 'bedrooms',
 'beds',
 'amenities',
 'price',
 'number_of_reviews',
 'review_scores_rating',
 'instant_bookable',
 'reviews_per_month',
 'room_type_hotel_room',
 'room_type_private_room',
 'room_type_shared_room',
 'host_response_time_ord',
 'bathrooms']

In [25]:
# Reorder columns and put them in categories that go together #
new_column_order = [
                    # identifiers
                    'id', 'host_id',
                    # listing basics
                    'room_type_hotel_room','room_type_private_room','room_type_shared_room',
                    'accommodates','bedrooms','beds','bathrooms',
                    # host info
                    'host_is_superhost', 'host_response_time_ord',
                    # pricing
                    'price',
                    # location
                    'latitude',  'longitude',
                    # reviews
                    'number_of_reviews','reviews_per_month','review_scores_rating',
                    # booking
                    'instant_bookable',
                    # amenities
                    'amenities']

In [26]:
df = df[new_column_order]

In [27]:
df.head(3)

,id,host_id,room_type_hotel_room,room_type_private_room,room_type_shared_room,accommodates,bedrooms,beds,bathrooms,host_is_superhost,host_response_time_ord,price,latitude,longitude,number_of_reviews,reviews_per_month,review_scores_rating,instant_bookable,amenities
0,108061,320564,False,False,False,2,1.0,1.0,1.0,1.0,3.0,100.0,35.60670,-82.55563,92,0.66,4.51,0,"[""Coffee maker"", ""Cleaning products"", ""Extra p..."
1,155305,746673,False,False,False,2,1.0,1.0,1.0,0.0,3.0,100.0,35.57864,-82.59578,383,2.70,4.59,0,"[""Shampoo"", ""Hair dryer"", ""Coffee maker"", ""Pet..."
2,156805,746673,False,True,False,2,1.0,1.0,2.5,0.0,3.0,66.0,35.57864,-82.59578,67,0.48,4.52,1,"[""Fire extinguisher"", ""Refrigerator"", ""Microwa..."


### Check null values ###
* inspect null values to see which needs to be dropped or imputed
* superhost: 0 or 1
* bedrooms: may be a studio, hotel room, or loft (quick inspection)
* beds: relatively small number, may be a pull out bed (quick inspection)
* reviews: could be a newer listing with no bookings yet, or could be a poorly performing listing. Will remove. 


In [28]:
df.isnull().sum()[df.isnull().sum() > 0] # check for null values

bedrooms                22500
beds                     3611
host_is_superhost         958
reviews_per_month       56360
review_scores_rating    56360
dtype: int64

### Null values: reviews ###

* Brand new listings: no reviews yet because they haven't had time to accumulate any. Nulls are noise.
* Established listings with no traction: been active, but guests aren't booking or aren't leaving reviews. Nulls might be negative signal for attractiveness.
* Will imput null values as 0

In [29]:
df['reviews_per_month'] = df['reviews_per_month'].fillna(0) # impute with 0
df['reviews_per_month'].isnull().sum()  # check for null values

np.int64(0)

In [30]:
df['review_scores_rating'] = df['review_scores_rating'].fillna(0) # impute with 0
df['review_scores_rating'].isnull().sum()  # check for null values

np.int64(0)

In [31]:
df['bedrooms'] = df['bedrooms'].fillna(0) # impute with 0 since there is no private bedroom (studio or shared)
df['bedrooms'].isnull().sum()  # check for null values

np.int64(0)

In [32]:
df['beds'] = df['beds'].fillna(1) # impute with 1, some are errors, but others are campsites or other types of lodging.
df['beds'].isnull().sum()  # check for null values

np.int64(0)

In [33]:
df.isnull().sum()[df.isnull().sum() > 0] # check for null values

host_is_superhost    958
dtype: int64

In [34]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


# explore listings and host when these conditions are met to see if's consistent with superhost status
df_super = df[
    (df['host_is_superhost'].isna()) & 
    (df['number_of_reviews'] > 200) &
    (df['reviews_per_month'] > 5) &
    (df['review_scores_rating'] > 4.5) &
    (df['host_response_time_ord'] == 3)]

df_super

,id,host_id,room_type_hotel_room,room_type_private_room,room_type_shared_room,accommodates,bedrooms,beds,bathrooms,host_is_superhost,host_response_time_ord,price,latitude,longitude,number_of_reviews,reviews_per_month,review_scores_rating,instant_bookable,amenities
17604,20468639,146144535,False,True,False,2,0.0,1.0,1.0,NaN,3.0,82.0,45.685760,-111.055650,452,6.48,4.71,0,"[""Room-darkening shades"", ""Laundromat nearby"",..."
60559,26775608,3583143,False,True,False,2,0.0,1.0,1.0,NaN,3.0,43.0,32.865610,-96.841560,363,6.19,5.00,0,"[""Heating"", ""Private living room"", ""Dedicated ..."
60623,27546256,58898579,False,True,False,2,0.0,1.0,1.0,NaN,3.0,32.0,32.801690,-96.771580,341,5.87,4.96,1,"[""Heating"", ""Children\u2019s dinnerware"", ""Bab..."
60664,28465345,132490994,False,True,False,2,0.0,2.0,1.0,NaN,3.0,45.0,32.809540,-96.766480,288,5.10,4.92,1,"[""Heating"", ""Conditioner"", ""Wifi"", ""Smoke alar..."
60748,33198135,249960325,False,False,False,3,1.0,2.0,1.0,NaN,3.0,70.0,32.793370,-96.758220,713,14.56,4.85,0,"[""Heating"", ""Stove"", ""Pets allowed"", ""Wifi"", ""..."
60811,33474953,252220275,False,True,False,3,0.0,2.0,1.0,NaN,3.0,49.0,32.803940,-96.754110,498,9.95,4.92,1,"[""Conditioner"", ""Single level home"", ""Dedicate..."
60835,34000242,252220275,False,False,False,2,1.0,1.0,1.0,NaN,3.0,86.0,32.805990,-96.756080,668,14.25,4.95,0,"[""Free driveway parking on premises"", ""Babysit..."
60975,38462630,118912877,False,True,False,2,0.0,1.0,1.0,NaN,3.0,55.0,32.732180,-96.861260,277,6.29,4.87,1,"[""Children\u2019s dinnerware"", ""Babysitter rec..."
61077,38907680,2791533,False,True,False,2,1.0,1.0,1.0,NaN,3.0,81.0,32.809780,-96.784070,473,10.74,4.86,0,"[""Heating"", ""Pets allowed"", ""Wifi"", ""Free drye..."
61082,38907909,2791533,False,True,False,2,0.0,1.0,1.0,NaN,3.0,50.0,32.811190,-96.783700,289,6.78,4.89,0,"[""Heating"", ""Conditioner"", ""Pets allowed"", ""Wi..."


### Superhost exploration was inconclusive, so will set any null values to 0

In [35]:
df['host_is_superhost'] = df['host_is_superhost'].fillna(0) # impute with 0
df['host_is_superhost'].isnull().sum()  # check for null values

np.int64(0)

In [36]:
df.isnull().sum()[df.isnull().sum() > 0] # check for null values

Series([], dtype: int64)

### Clean Amenities Data

1. Inspect raw amenities to understand the data structure
2. Strings into Python list (remove brackets, quotes, extra spacing)
3. Normalize strings: lowercase, strip whitespace
4. Map raw strings to categories using regex
   - e.g. "HDTV", "Roku TV", "Amazon Firestick" → `tv`
   - e.g. "Coffee machine", "coffee maker", "Keurig" → `coffee_maker`
5. Check frequency of amenities
6. Expand list into binary columns (one per amenity category) - 1: included 0: not included

In [37]:
df['amenities'].head()

0    ["Coffee maker", "Cleaning products", "Extra p...
1    ["Shampoo", "Hair dryer", "Coffee maker", "Pet...
2    ["Fire extinguisher", "Refrigerator", "Microwa...
3    ["Shampoo", "Electric stove", "Hair dryer", "C...
4    ["Shampoo", "Hair dryer", "Coffee maker", "Cle...
Name: amenities, dtype: object

In [38]:
import ast
import re

def parse_amenities(raw):
    try:
        parsed = ast.literal_eval(raw)
        return [item.strip().lower() for item in parsed if isinstance(item, str) and item.strip()]
    except:
        cleaned = re.sub(r'[\[\]"\\]', '', str(raw))  # remove brackets, quotes, backslashes, strip all the junk and split by comma
        return [item.strip().lower() for item in cleaned.split(',') if item.strip()]

df['amenities_parsed'] = df['amenities'].apply(parse_amenities)

Had some issues cleaning the strings, somehow brackets and backslashes kept showing up

In [39]:
# verify it worked
print(type(df['amenities_parsed'].iloc[0]))  # should say <class 'list'>
print(df['amenities_parsed'].iloc[0])        

<class 'list'>
['coffee maker', 'cleaning products', 'extra pillows and blankets', 'pets allowed', 'cooking basics', 'indoor fireplace: gas', 'dishes and silverware', 'private entrance', 'free street parking', 'essentials', 'single level home', 'free parking on premises', 'freezer', 'dining table', 'refrigerator', 'free dryer – in unit', 'induction stove', 'wifi', 'carbon monoxide alarm', 'blender', 'hangers', 'central air conditioning', 'hot water kettle', 'microwave', 'books and reading material', 'patio or balcony', 'long term stays allowed', 'central heating', 'hot water', 'room-darkening shades', 'toaster', 'kitchen', 'wine glasses', 'ceiling fan', 'host greets you', 'smoke alarm', 'baking sheet', 'bbq grill', 'courtyard view', 'oven', 'dishwasher', 'clothing storage', 'backyard', 'bed linens', 'outdoor dining area', 'free washer – in unit', 'iron']


In [40]:
# IMPORTANT
# Count all the amenities, then export and review csv file to determine categories for amenity mapping

from collections import Counter

# Create frequency count of amenities
all_amenities = [item for sublist in df['amenities_parsed'] for item in sublist]
freq = pd.DataFrame(Counter(all_amenities).most_common(), columns=['amenity', 'count'])

print(f'Total unique amenities: {len(freq)}')

# Fix encoding issues by replacing or removing problematic characters
freq['amenity'] = freq['amenity'].apply(lambda x: x.encode('utf-8', errors='replace').decode('utf-8'))

Total unique amenities: 40611


In [41]:
 freq.tail() # these should all be categorized into standard amenities

,amenity,count
40606,"65"" hdtv with standard cable, amazon prime vid...",1
40607,fast wifi – 916 mbps,1
40608,"40"" hdtv with fire tv, hbo max, netflix",1
40609,"hdtv with amazon prime video, disney+, fire tv...",1
40610,dove/ suave body soap,1


In [42]:
# explore CSV file on sheets to see potential amenity categories

freq.to_csv('amenity_freq_parsed.csv', index=False) 

### Map the Amenities into new categories

In [43]:
AMENITY_MAP = {
    # internet
    r'wi.?fi|internet': 'wifi',
    
    # tv / streaming - catches size/brand/service combos
    r'\btv\b|television|hdtv|smart tv|roku|firestick|fire tv|apple tv|chromecast|amazon prime video|disney\+|hbo|netflix|hulu|cable|streaming': 'tv',
    
    # kitchen
    r'\bkitchen\b': 'kitchen',
    r'\bdishwasher\b': 'dishwasher',
    r'\brefrigerator\b|fridge|freezer': 'refrigerator',
    r'\bmicrowave\b': 'microwave',
    r'\boven\b': 'oven',
    r'\bstove\b|cooktop|induction stove|hot plate|electric burner': 'stove',
    r'coffee|keurig|nespresso': 'coffee_maker',
    r'blender|vitamix': 'blender',
    r'toaster|airfyer': 'toaster',
    r'cooking basics|dishes and silverware|pots and pans|baking sheet|wine glasses|dining table': 'cooking_basics',
    
    # laundry - dishwasher and hair dryer must come before washer/dryer so it doesn't get counted incorrectly
    r'\bdishwasher\b': 'dishwasher',
    r'\bhair dryer\b': 'hair_dryer',
    r'\bwasher\b|washing machine|laundry': 'washer',
    r'\bdryer\b|free dryer': 'dryer',
    
    # ac, heating, etc
    r'air conditioning|central air|\bac\b|a\.c\.': 'air_conditioning',
    r'\bheating\b|central heating': 'heating',
    r'ceiling fan|\bfan\b': 'fan',
    
    # bedroom/bathroom
    r'bed linen|linens|extra pillow|extra blanket': 'bed_linens',
    r'\bhangers\b': 'hangers',
    r'\biron\b': 'iron',
    r'soap|shampoo|conditioner|body wash|toiletries|body soap': 'toiletries',
    r'\bhair dryer\b': 'hair_dryer',
    r'hot water': 'hot_water',
    r'bathtub|bath tub|soaking tub': 'bathtub',
    
    # safety
    r'smoke alarm|smoke detector': 'smoke_alarm',
    r'carbon monoxide': 'carbon_monoxide_alarm',
    r'fire extinguisher': 'fire_extinguisher',
    r'first aid': 'first_aid_kit',
    
    # outdoor/recreation
    r'\bpool\b': 'pool',
    r'hot tub|jacuzzi|whirlpool': 'hot_tub',
    r'bbq|barbecue|\bgrill\b': 'bbq_grill',
    r'beach access|beachfront|beach front': 'beach_access',
    r'gym|fitness|exercise equipment|free weights': 'gym',
    r'fireplace': 'fireplace',
    r'patio|balcony|terrace|lanai': 'patio_balcony',
    r'backyard|garden\b|outdoor space': 'outdoor_space',
    r'ski.?in|ski.?out': 'ski_access',
    
    # views
    r'ocean view|sea view|bay view|water view': 'view_water',
    r'city view|skyline view|cityscape': 'view_city',
    r'mountain view|valley view|desert view|golf.*view|nature view': 'view_nature',
    r'garden view|courtyard view|park view|pool view|resort view': 'view_other',
    
    # parking
    r'free.*parking|parking.*free|free street parking|driveway': 'parking_free',
    r'paid.*parking|parking.*paid|valet|parking.*fee': 'parking_paid',
    
    # convenience/access
    r'ev.?charger|electric.*vehicle': 'ev_charger',
    r'self check.?in|keypad|lockbox|smartlock|smart lock': 'self_checkin',
    r'workspace|dedicated.*work|\bdesk\b': 'workspace',
    r'long.?term|long term stays': 'long_term_stays',
    r'\bessentials\b': 'essentials',
    r'private entrance': 'private_entrance',
    r'luggage dropoff|early check|late check|flexible': 'flexible_checkin',
    
    # entertainment
    r'sound system|bluetooth.*speaker|sonos|\bspeaker': 'sound_system',
    r'game console|nintendo|playstation|xbox': 'game_console',
    r'record player|vinyl|piano|\bguitar\b': 'musical_instruments',
    
    # kids
    r'crib|pack.?n.?play|baby.*bed|high chair|children|kids|toys|board games|books and': 'kids_amenities',
    
    # host services
    r'host greets|concierge|breakfast included|chef': 'host_services',
    r'cleaning products|cleaning supplies': 'cleaning_supplies',
}

In [44]:
def map_amenities(amenity_list):
    found = set()
    for item in amenity_list:
        for pattern, label in AMENITY_MAP.items():
            if re.search(pattern, item, re.IGNORECASE):
                found.add(label)
                break
    return list(found)

In [45]:
df['amenities_clean'] = df['amenities_parsed'].apply(map_amenities)

In [46]:
# verify
print(df['amenities_clean'].apply(len).describe())
df['amenities_clean'].head(3)

count    268387.000000
mean         25.981434
std           9.379996
min           0.000000
25%          19.000000
50%          27.000000
75%          33.000000
max          54.000000
Name: amenities_clean, dtype: float64


0    [kids_amenities, outdoor_space, heating, micro...
1    [heating, microwave, stove, kitchen, hangers, ...
2    [outdoor_space, essentials, refrigerator, hot_...
Name: amenities_clean, dtype: object

### Amenities Summary

mean: 25.9   the average listing matched 26 categories

min: 0   some listings matched nothing (need to look into this)

max: 54  map has ~35 categories, so 54 is basically means a listing"has almost everything"

50%: 27   half of listings have 27+ amenities matched

In [47]:
# inspect the listing with the most amenities
idx = df['amenities_clean'].apply(len).idxmax()
print(df['amenities_clean'].iloc[idx])
print(f'\nRaw count: {len(df["amenities_clean"].iloc[idx])}')

['kids_amenities', 'outdoor_space', 'workspace', 'flexible_checkin', 'view_water', 'heating', 'microwave', 'washer', 'blender', 'stove', 'kitchen', 'first_aid_kit', 'hangers', 'bbq_grill', 'fan', 'toiletries', 'toaster', 'patio_balcony', 'game_console', 'bathtub', 'tv', 'view_other', 'beach_access', 'carbon_monoxide_alarm', 'view_nature', 'parking_free', 'cooking_basics', 'musical_instruments', 'fireplace', 'hot_tub', 'dishwasher', 'refrigerator', 'ev_charger', 'long_term_stays', 'dryer', 'parking_paid', 'fire_extinguisher', 'gym', 'sound_system', 'cleaning_supplies', 'iron', 'private_entrance', 'self_checkin', 'bed_linens', 'essentials', 'air_conditioning', 'hot_water', 'hair_dryer', 'smoke_alarm', 'oven', 'wifi', 'view_city', 'coffee_maker', 'pool']

Raw count: 54


In [48]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
amenity_dummies = pd.DataFrame(
    mlb.fit_transform(df['amenities_clean']),
    columns=[f'amenity_{c}' for c in mlb.classes_],
    index=df.index)

In [49]:
# join back and drop working columns
df = pd.concat([df, amenity_dummies], axis=1)
df = df.drop(columns=['amenities_parsed', 'amenities_clean', 'amenities'])

In [50]:
# convert amenity columns from float to int
amenity_cols = [c for c in df.columns if c.startswith('amenity_')]
df[amenity_cols] = df[amenity_cols].astype(int)

In [51]:
# verify
print(df.shape)
amenity_dummies.sum().sort_values(ascending=False)

(268387, 74)


amenity_wifi                     263384
amenity_smoke_alarm              252143
amenity_kitchen                  239734
amenity_tv                       237772
amenity_essentials               234797
amenity_air_conditioning         224928
amenity_hangers                  212656
amenity_heating                  210827
amenity_hair_dryer               209333
amenity_carbon_monoxide_alarm    209193
amenity_refrigerator             205397
amenity_cooking_basics           204977
amenity_hot_water                202033
amenity_toiletries               201896
amenity_iron                     201649
amenity_parking_free             201387
amenity_coffee_maker             191667
amenity_washer                   191481
amenity_microwave                183379
amenity_bed_linens               179009
amenity_dryer                    178127
amenity_fire_extinguisher        175773
amenity_self_checkin             158101
amenity_stove                    158009
amenity_oven                     156217


In [52]:
# 1. check for any remaining nulls
df.isnull().sum()[df.isnull().sum() > 0]

Series([], dtype: int64)

In [53]:
# 2. check dtypes - everything should be numeric except id/host_id
df.dtypes

id                                 int64
host_id                            int64
room_type_hotel_room                bool
room_type_private_room              bool
room_type_shared_room               bool
accommodates                       int64
bedrooms                         float64
beds                             float64
bathrooms                        float64
host_is_superhost                float64
host_response_time_ord           float64
price                            float64
latitude                         float64
longitude                        float64
number_of_reviews                  int64
reviews_per_month                float64
review_scores_rating             float64
instant_bookable                   int64
amenity_air_conditioning           int64
amenity_bathtub                    int64
amenity_bbq_grill                  int64
amenity_beach_access               int64
amenity_bed_linens                 int64
amenity_blender                    int64
amenity_carbon_m

In [54]:
# 3. check shape and preview
print(df.shape)
df.head(3)

(268387, 74)


,id,host_id,room_type_hotel_room,room_type_private_room,room_type_shared_room,accommodates,bedrooms,beds,bathrooms,host_is_superhost,host_response_time_ord,price,latitude,longitude,number_of_reviews,reviews_per_month,review_scores_rating,instant_bookable,amenity_air_conditioning,amenity_bathtub,amenity_bbq_grill,amenity_beach_access,amenity_bed_linens,amenity_blender,amenity_carbon_monoxide_alarm,amenity_cleaning_supplies,amenity_coffee_maker,amenity_cooking_basics,amenity_dishwasher,amenity_dryer,amenity_essentials,amenity_ev_charger,amenity_fan,amenity_fire_extinguisher,amenity_fireplace,amenity_first_aid_kit,amenity_flexible_checkin,amenity_game_console,amenity_gym,amenity_hair_dryer,amenity_hangers,amenity_heating,amenity_host_services,amenity_hot_tub,amenity_hot_water,amenity_iron,amenity_kids_amenities,amenity_kitchen,amenity_long_term_stays,amenity_microwave,amenity_musical_instruments,amenity_outdoor_space,amenity_oven,amenity_parking_free,amenity_parking_paid,amenity_patio_balcony,amenity_pool,amenity_private_entrance,amenity_refrigerator,amenity_self_checkin,amenity_ski_access,amenity_smoke_alarm,amenity_sound_system,amenity_stove,amenity_toaster,amenity_toiletries,amenity_tv,amenity_view_city,amenity_view_nature,amenity_view_other,amenity_view_water,amenity_washer,amenity_wifi,amenity_workspace
0,108061,320564,False,False,False,2,1.0,1.0,1.0,1.0,3.0,100.0,35.60670,-82.55563,92,0.66,4.51,0,1,0,1,0,1,1,1,1,1,1,1,1,1,0,1,0,1,0,0,0,0,0,1,1,1,0,1,1,1,1,1,1,0,1,1,1,0,1,0,1,1,0,0,1,0,1,1,0,0,0,0,1,0,1,1,0
1,155305,746673,False,False,False,2,1.0,1.0,1.0,0.0,3.0,100.0,35.57864,-82.59578,383,2.70,4.59,0,1,0,0,0,0,0,0,0,1,1,0,0,1,0,0,1,0,0,0,0,0,1,1,1,0,0,1,1,0,1,0,1,0,0,1,1,0,1,0,1,1,1,0,1,0,1,0,1,0,0,0,0,0,0,1,0
2,156805,746673,False,True,False,2,1.0,1.0,2.5,0.0,3.0,66.0,35.57864,-82.59578,67,0.48,4.52,1,0,0,0,0,0,0,1,0,1,1,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,1,0,1,0,1,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0


## Update Room Types to 0 or 1

In [55]:
# convert boolean room type columns to int
bool_cols = ['room_type_hotel_room', 'room_type_private_room', 'room_type_shared_room']
df[bool_cols] = df[bool_cols].astype(int)

In [56]:
# verify
df[bool_cols].head(3)

,room_type_hotel_room,room_type_private_room,room_type_shared_room
0,0,0,0
1,0,0,0
2,0,1,0


In [57]:
print(df.shape)
print(df.dtypes)
df.isnull().sum()[df.isnull().sum() > 0]

(268387, 74)
id                                 int64
host_id                            int64
room_type_hotel_room               int64
room_type_private_room             int64
room_type_shared_room              int64
accommodates                       int64
bedrooms                         float64
beds                             float64
bathrooms                        float64
host_is_superhost                float64
host_response_time_ord           float64
price                            float64
latitude                         float64
longitude                        float64
number_of_reviews                  int64
reviews_per_month                float64
review_scores_rating             float64
instant_bookable                   int64
amenity_air_conditioning           int64
amenity_bathtub                    int64
amenity_bbq_grill                  int64
amenity_beach_access               int64
amenity_bed_linens                 int64
amenity_blender                    int64
ame

Series([], dtype: int64)

In [58]:
df.head()

,id,host_id,room_type_hotel_room,room_type_private_room,room_type_shared_room,accommodates,bedrooms,beds,bathrooms,host_is_superhost,host_response_time_ord,price,latitude,longitude,number_of_reviews,reviews_per_month,review_scores_rating,instant_bookable,amenity_air_conditioning,amenity_bathtub,amenity_bbq_grill,amenity_beach_access,amenity_bed_linens,amenity_blender,amenity_carbon_monoxide_alarm,amenity_cleaning_supplies,amenity_coffee_maker,amenity_cooking_basics,amenity_dishwasher,amenity_dryer,amenity_essentials,amenity_ev_charger,amenity_fan,amenity_fire_extinguisher,amenity_fireplace,amenity_first_aid_kit,amenity_flexible_checkin,amenity_game_console,amenity_gym,amenity_hair_dryer,amenity_hangers,amenity_heating,amenity_host_services,amenity_hot_tub,amenity_hot_water,amenity_iron,amenity_kids_amenities,amenity_kitchen,amenity_long_term_stays,amenity_microwave,amenity_musical_instruments,amenity_outdoor_space,amenity_oven,amenity_parking_free,amenity_parking_paid,amenity_patio_balcony,amenity_pool,amenity_private_entrance,amenity_refrigerator,amenity_self_checkin,amenity_ski_access,amenity_smoke_alarm,amenity_sound_system,amenity_stove,amenity_toaster,amenity_toiletries,amenity_tv,amenity_view_city,amenity_view_nature,amenity_view_other,amenity_view_water,amenity_washer,amenity_wifi,amenity_workspace
0,108061,320564,0,0,0,2,1.0,1.0,1.0,1.0,3.0,100.0,35.60670,-82.55563,92,0.66,4.51,0,1,0,1,0,1,1,1,1,1,1,1,1,1,0,1,0,1,0,0,0,0,0,1,1,1,0,1,1,1,1,1,1,0,1,1,1,0,1,0,1,1,0,0,1,0,1,1,0,0,0,0,1,0,1,1,0
1,155305,746673,0,0,0,2,1.0,1.0,1.0,0.0,3.0,100.0,35.57864,-82.59578,383,2.70,4.59,0,1,0,0,0,0,0,0,0,1,1,0,0,1,0,0,1,0,0,0,0,0,1,1,1,0,0,1,1,0,1,0,1,0,0,1,1,0,1,0,1,1,1,0,1,0,1,0,1,0,0,0,0,0,0,1,0
2,156805,746673,0,1,0,2,1.0,1.0,2.5,0.0,3.0,66.0,35.57864,-82.59578,67,0.48,4.52,1,0,0,0,0,0,0,1,0,1,1,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,1,0,1,0,1,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0
3,197263,961396,0,1,0,2,1.0,1.0,1.0,1.0,2.0,45.0,35.57808,-82.63689,66,0.51,4.95,0,1,1,0,0,1,1,0,1,0,1,0,1,1,0,1,1,0,1,0,0,0,1,1,1,0,0,1,0,1,1,0,1,0,0,1,1,0,1,0,0,1,0,0,1,0,1,0,1,1,0,0,0,0,1,1,1
4,209068,1029919,0,0,0,4,1.0,2.0,1.0,1.0,2.0,100.0,35.61856,-82.55276,60,0.43,4.87,0,1,0,1,0,1,0,1,1,1,1,1,1,1,0,1,1,0,0,1,0,0,1,1,1,0,0,1,1,1,1,1,1,0,0,1,1,0,1,0,1,1,0,0,1,0,1,0,1,1,0,0,0,0,1,1,1


In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 268387 entries, 0 to 268386
Data columns (total 74 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   id                             268387 non-null  int64  
 1   host_id                        268387 non-null  int64  
 2   room_type_hotel_room           268387 non-null  int64  
 3   room_type_private_room         268387 non-null  int64  
 4   room_type_shared_room          268387 non-null  int64  
 5   accommodates                   268387 non-null  int64  
 6   bedrooms                       268387 non-null  float64
 7   beds                           268387 non-null  float64
 8   bathrooms                      268387 non-null  float64
 9   host_is_superhost              268387 non-null  float64
 10  host_response_time_ord         268387 non-null  float64
 11  price                          268387 non-null  float64
 12  latitude                      

In [60]:
clean_data_path = os.path.join('..', 'data', 'processed', 'airbnb_clean.csv')
df.to_csv(clean_data_path, index=False)
print(f'Saved → {clean_data_path}')

Saved → ..\data\processed\airbnb_clean.csv


## Data Wrangling Summary

- Combined detailed Airbnb listing data across multiple cities into a single dataframe (268,387 listings)
- Dropped irrelevant and redundant columns, retaining only features applicable to pricing and booking analysis
- Converted raw string columns to numeric types (price, response rate, superhost status, instant bookable)
- Encoded categorical variables: room type → binary dummy columns, host response time → ordinal scale (0–3)
- Extracted bathroom count from free-text descriptions, imputed missing values based on listing context
- Inspected and imputed null values across review scores, bed/bedroom counts, and host attributes
- Parsed and standardized 40,611 unique raw amenity strings into 54 categories using regex mapping
- Expanded amenity categories into binary feature columns (1 = present, 0 = not present)